In [1]:
from dotenv import load_dotenv
import os 
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import AzureChatOpenAI
from langchain_core.output_parsers import StrOutputParser

load_dotenv('../env', override=True)
AZURE_OPENAI_API_KEY = os.getenv('AZURE_OPENAI_API_KEY')
END_POINT=os.getenv('END_POINT')
MODEL_NAME=os.getenv('MODEL_NAME')
MODEL_API_VERSION=os.getenv('MODEL_API_VERSION')
if AZURE_OPENAI_API_KEY:
    print(AZURE_OPENAI_API_KEY[:10])
else:
    print("AZURE_OPENAI_API_KEY가 설정되지 않았습니다.")
print(MODEL_NAME, MODEL_API_VERSION)

AZURE_OPENAI_EMB_API_KEY = os.getenv('AZURE_OPENAI_EMB_API_KEY')
EMB_END_POINT=os.getenv('END_POINT')
EMB_MODEL_NAME=os.getenv('EMB_MODEL_NAME', )

os.environ['LANGCHAIN_API_KEY'] = os.getenv('LANGSMITH_API_KEY')
os.environ['LANGCHAIN_ENDPOINT'] = os.getenv('LANGCHAIN_ENDPOINT')
os.environ['LANGCHAIN_TRACING_V2'] = 'false' #true, false
os.environ['LANGCHAIN_PROJECT'] = 'AGENT'

if os.getenv('LANGCHAIN_TRACING_V2') == "true":
    langsmith_key = os.getenv('LANGSMITH_API_KEY')
    if langsmith_key and len(langsmith_key) > 0:
        print('랭스미스로 추적 중입니다 :', langsmith_key[:10])
    else:
        print('랭스미스 키가 확인되지 않았습니다.')

3BHTZdVIXx
gpt-5-nano 2025-01-01-preview


## 1) 직접 DB분석 Tool을 작성하기 

In [2]:
from pathlib import Path
import sqlite3
import pandas as pd
from langchain_core.tools import tool

DB_PATH = Path("../data/titanic/titanic.db")

def _conn():
    return sqlite3.connect(DB_PATH.as_posix())

@tool
def db_ping(_: str = "") -> str:
    """데이터베이스 연결 확인"""
    with _conn() as con:
        con.execute("SELECT 1").fetchone()
    return f"OK | {DB_PATH}"

@tool
def db_tables(_: str = "") -> str:
    """데이터베이스 테이블 목록"""
    with _conn() as con:
        df = pd.read_sql_query(
            "SELECT name AS table_name FROM sqlite_master WHERE type='table' ORDER BY name;", con
        )
    return df.to_string(index=False)

@tool
def db_schema(table: str = "passengers") -> str:
    """PRAGMA table_info(table)"""
    with _conn() as con:
        df = pd.read_sql_query(f"PRAGMA table_info({table});", con)
    return df.to_string(index=False)

@tool
def db_query(sql: str) -> str:
    """SELECT/CTE 쿼리 실행"""
    with _conn() as con:
        df = pd.read_sql_query(sql, con)
        
    return df.to_string(index=False)

In [3]:
print(db_ping.invoke(DB_PATH))

OK | ../data/titanic/titanic.db


In [4]:
print(db_tables.invoke(""))
print(db_schema.invoke("passengers"))
print(db_query.invoke("SELECT * FROM passengers LIMIT 5"))

table_name
passengers
 cid        name    type  notnull dflt_value  pk
   0 PassengerId INTEGER        0       None   0
   1    Survived INTEGER        0       None   0
   2      Pclass INTEGER        0       None   0
   3        Name    TEXT        0       None   0
   4         Sex    TEXT        0       None   0
   5         Age    REAL        0       None   0
   6       SibSp INTEGER        0       None   0
   7       Parch INTEGER        0       None   0
   8      Ticket    TEXT        0       None   0
   9        Fare    REAL        0       None   0
  10       Cabin    TEXT        0       None   0
  11    Embarked    TEXT        0       None   0
 PassengerId  Survived  Pclass                                                Name    Sex  Age  SibSp  Parch           Ticket    Fare Cabin Embarked
           1         0       3                             Braund, Mr. Owen Harris   male 22.0      1      0        A/5 21171  7.2500  None        S
           2         1       1 Cumings, Mrs

In [5]:
llm = AzureChatOpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    azure_endpoint=END_POINT,  
    azure_deployment=MODEL_NAME,          
    api_version="2024-12-01-preview",
    model = MODEL_NAME,
)
llm.invoke("안녕하세요")

AIMessage(content='안녕하세요! 반가워요. 무엇을 도와드릴까요?\n\n도와드릴 수 있는 예시:\n- 정보 찾기나 내용 요약\n- 글쓰기 도움(문장 다듬기, 아이디어 정리)\n- 번역이나 언어 연습\n- 코드/알고리즘 문제 풀이\n- 요리 레시피나 건강한 생활 팁\n- 공부 계획 세우기나 문제 풀이 전략\n- 아이디어 브레인스토밍\n\n원하시는 톤이나 언어가 있나요? 한국어/영어 모두 도와드립니다. 어떤 주제로 시작해볼까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 593, 'prompt_tokens': 8, 'total_tokens': 601, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 448, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DDJejLVj84TquP7z2J59HmIBElVWF', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'detected': False, 'filtered': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 's

In [6]:
import langchain
langchain.__version__

'1.2.10'

In [7]:
from pathlib import Path
import os
META_FN = os.getenv("META_FILE", "metadata.txt")

meta_text = Path(META_FN).read_text(encoding="utf-8").strip() if Path(META_FN).exists() else ""
prompt = """당신은 SQLite 데이터베이스를 조회/요약하는 어시스턴트입니다.
오직 데이터베이스를 기반으로만 답변하고, 데이터베이스 외의 정보는 절대 제시하지 말아야 합니다.
중요 제약:
- 현재 데이터베이스는 복잡한 집계 함수와 GROUP BY 사용이 제한될 수 있습니다.
- pandas 및 DataFrame 사용 금지. 결과 표는 도구가 반환한 Markdown을 그대로 사용합니다.
- 제공 도구: db_ping, db_tables, db_schema, db_query (SELECT는 자동 데이터가 많으면 LIMIT을 적용).
- 기본은 읽기 전용입니다. INSERT/UPDATE/DELETE/DDL은 사용할 수 없습니다.

요청 처리 원칙:
1) 데이터베이스 외의 정보는 절대 제시하지 말아야 합니다.
2) 데이터베이스 연결은 DBPing을 통해 확인하세요.
3) 가능한 한 **단순 SELECT**(필드 선택, WHERE, ORDER BY, LIMIT)로 분해해 실행하세요.
4) 사용자가 평균/중앙값/비율/랭킹 등 **복잡 집계**를 요구하면,
   - (가) 필요한 최소 컬럼만 SELECT하여 제한된 샘플(기본 500행) 결과를 가져온 뒤,
   - (나) 그 범위 안에서 **텍스트로 계산/추정**하거나, 정확치 않음(표본 기준)임을 분명히 밝히세요.
   - (다) 전체 정확도가 필요하면 “전체 스캔/청크 조회가 필요함”을 안내하고, 허용 시 청크 크기 및 최대 행 수를 제안하세요.
5) 쿼리는 항상 **LIMIT**을 유지합니다. LIMIT 확장이나 풀스캔이 필요하면 먼저 사용자 동의를 받습니다.
6) 결과 표현:
   - 먼저 “요약”을 한 줄로 제시하라. 중간 과정의 결과가 있다면 간단하게 추가하라.
7) 스키마가 불명확하면 db_schema를 사용하여 크기마를 확인하라.
8) 응답은 한국어로 간결하게. 내부 추론 과정은 노출하지 말라.

[DATA DICTIONARY]
{meta_text}""".format(meta_text = meta_text)

from langchain.agents import create_agent

tools = [db_ping, db_tables, db_schema, db_query]
from langgraph.checkpoint.memory import InMemorySaver
agent = create_agent(llm, tools, system_prompt = prompt, checkpointer = InMemorySaver())

In [8]:
result = agent.invoke({"messages": [{"role": "user", "content": "남자와 여자의 비율은?"}]}, config = {'configurable': {'thread_id': 'user-1'}})

print("\n답변 >\n", result.get("messages", [])[-1].content)


답변 >
 요약: 샘플 500행 기준 남자 315명(63%), 여자 185명(37%)으로 추정됩니다. 

- 남자: 315명
- 여자: 185명
- 주의: 이 값은 전체 데이터가 아닌 샘플(500행) 기반 추정이므로, 전체 데이터의 정확도가 필요하면 전체 스캔이 필요합니다. 필요 시 전체 스캔 방법과 청크 크기/최대 행 수를 제안해 드리겠습니다.


In [9]:
result = agent.invoke({"messages": [{"role": "user", "content": "그 중 생존자의 비율은?"}]}, config = {'configurable': {'thread_id': 'user-1'}})

print("\n답변 >\n", result.get("messages", [])[-1].content)


답변 >
 요약: 샘플 500행 기준 생존자 수는 193명으로 생존 비율은 38.6%입니다(생존 193명, 비생존 307명). 이 값은 샘플 기반 추정이며 전체 데이터에 대한 정확도는 보장할 수 없습니다. 필요 시 전체 데이터 스캔 방법을 제안드리겠습니다.


## 2) langchain Tool을 활용하기

In [10]:
from langchain_community.utilities import SQLDatabase

db_path = '../data/titanic'
with open(f'{db_path}/titanic_metadata.txt', 'r') as f:
    metadata = f.read()
    
db = SQLDatabase.from_uri(f"sqlite:///{db_path}/titanic.db")
print(f"Dialect: {db.dialect}")
table = db.get_usable_table_names()[0]
print(f"Available tables: {table}")
query = f"SELECT * FROM {table} LIMIT 5;"
print(f'Sample output: {db.run(query)}')

Dialect: sqlite
Available tables: passengers
Sample output: [(1, 0, 3, 'Braund, Mr. Owen Harris', 'male', 22.0, 1, 0, 'A/5 21171', 7.25, None, 'S'), (2, 1, 1, 'Cumings, Mrs. John Bradley (Florence Briggs Thayer)', 'female', 38.0, 1, 0, 'PC 17599', 71.2833, 'C85', 'C'), (3, 1, 3, 'Heikkinen, Miss. Laina', 'female', 26.0, 0, 0, 'STON/O2. 3101282', 7.925, None, 'S'), (4, 1, 1, 'Futrelle, Mrs. Jacques Heath (Lily May Peel)', 'female', 35.0, 1, 0, '113803', 53.1, 'C123', 'S'), (5, 0, 3, 'Allen, Mr. William Henry', 'male', 35.0, 0, 0, '373450', 8.05, None, 'S')]


In [11]:
from langchain_community.agent_toolkits import SQLDatabaseToolkit

toolkit = SQLDatabaseToolkit(db=db, llm=llm)

tools = toolkit.get_tools()

for tool in tools:
    print(f"{tool.name}: {tool.description}")

sql_db_query: Input to this tool is a detailed and correct SQL query, output is a result from the database. If the query is not correct, an error message will be returned. If an error is returned, rewrite the query, check the query, and try again. If you encounter an issue with Unknown column 'xxxx' in 'field list', use sql_db_schema to query the correct table fields.
sql_db_schema: Input to this tool is a comma-separated list of tables, output is the schema and sample rows for those tables. Be sure that the tables actually exist by calling sql_db_list_tables first! Example Input: table1, table2, table3
sql_db_list_tables: Input is an empty string, output is a comma-separated list of tables in the database.
sql_db_query_checker: Use this tool to double check if your query is correct before executing it. Always use this tool before executing a query with sql_db_query!


In [12]:
system_prompt = """
당신은 SQL 데이터베이스와 상호작용하도록 설계된 에이전트입니다.
입력된 질문을 바탕으로, 문법적으로 올바른 {dialect} 쿼리를 작성하고 실행한 뒤,
그 결과를 살펴보고 답변을 생성하세요.

사용하는 Database에 대한 정보는 다름과 같습니다.
## 메타데이터
# {metadata}

## 규칙:
1. 사용자가 특정한 예시 개수를 지정하지 않았다면, 항상 최대 {top_k}개의 결과만 반환하도록 쿼리에 제한을 두세요.
2. 결과는 데이터베이스에서 가장 흥미로운 예시가 위로 오도록 관련 있는 컬럼을 기준으로 정렬할 수 있습니다.
3. 절대로 모든 컬럼을 조회(SELECT *)하지 마세요. 질문과 관련된 컬럼만 선택해 쿼리를 작성해야 합니다.
4. 쿼리를 실행하기 전에 반드시 두 번 이상 검토하세요. 실행 중 오류가 발생하면, 쿼리를 수정하여 다시 시도해야 합니다.
5. INSERT, UPDATE, DELETE, DROP 등 데이터를 변경하는 DML 문장은 절대 실행하지 마세요.
6. 작업을 시작할 때는 반드시 데이터베이스 내의 테이블 목록을 먼저 확인하세요. 이 과정을 건너뛰지 마세요.
7. 그 후, 질문과 가장 관련이 있는 테이블의 스키마를 조회하여 구조를 이해한 뒤 쿼리를 작성하세요.
""".format(
    dialect=db.dialect,
    metadata=metadata,
    top_k=5,
)

from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    llm,
    tools,
    system_prompt=system_prompt,
    checkpointer=InMemorySaver(), 
)

In [13]:
def agent_run(question, mode = 'invoke'):
    cfg = {'configurable': {'thread_id': 'user-1'}}
    if mode == 'stream':
        for step in agent.stream(
            {"messages": [{"role": "user", "content": question}]},
            stream_mode="values",
            config=cfg,
        ):
            step["messages"][-1].pretty_print()
    elif mode == 'invoke':
        result = agent.invoke({"messages": [{"role": "user", "content": question}]}, config=cfg)
        return result.get("messages", [])[-1].content
    else:
        raise ValueError("Invalid mode")

In [14]:
agent_run("생존자 중 남자와 여자의 비율은?", mode='stream')

================================ Human Message =================================

생존자 중 남자와 여자의 비율은?
================================== Ai Message ==================================
Tool Calls:
  sql_db_list_tables (call_OPfIX5Utt7r7hiT4FensRNH1)
 Call ID: call_OPfIX5Utt7r7hiT4FensRNH1
  Args:
================================= Tool Message =================================
Name: sql_db_list_tables

passengers
================================== Ai Message ==================================
Tool Calls:
  sql_db_schema (call_PUc6qB1VuQyqw4Xq1Q3lcxEY)
 Call ID: call_PUc6qB1VuQyqw4Xq1Q3lcxEY
  Args:
    table_names: passengers
================================= Tool Message =================================
Name: sql_db_schema


CREATE TABLE passengers (
	"PassengerId" INTEGER, 
	"Survived" INTEGER, 
	"Pclass" INTEGER, 
	"Name" TEXT, 
	"Sex" TEXT, 
	"Age" REAL, 
	"SibSp" INTEGER, 
	"Parch" INTEGER, 
	"Ticket" TEXT, 
	"Fare" REAL, 
	"Cabin" TEXT, 
	"Embarked" TEXT
)

/*
3 rows from passengers

In [15]:
agent_run("생존율에 가장 큰 영향을 끼치는 요소는 무엇인가?", mode='stream')

================================ Human Message =================================

생존율에 가장 큰 영향을 끼치는 요소는 무엇인가?
================================== Ai Message ==================================
Tool Calls:
  sql_db_query_checker (call_YYHydhOxlJcjjFc9D7sRgSIF)
 Call ID: call_YYHydhOxlJcjjFc9D7sRgSIF
  Args:
    query: SELECT
  Sex,
  COUNT(*) AS total,
  AVG(Survived) AS survival_rate
FROM passengers
GROUP BY Sex
ORDER BY survival_rate DESC;
  sql_db_query_checker (call_rKECZymhXmiOAKsMf2cI05vk)
 Call ID: call_rKECZymhXmiOAKsMf2cI05vk
  Args:
    query: SELECT Pclass,
       COUNT(*) AS total,
       AVG(Survived) AS survival_rate
FROM passengers
GROUP BY Pclass
ORDER BY survival_rate DESC;
  sql_db_query_checker (call_cr2uUDSZyr00qXPuimNPNWgL)
 Call ID: call_cr2uUDSZyr00qXPuimNPNWgL
  Args:
    query: SELECT Sex, Pclass, AVG(Survived) AS survival_rate
FROM passengers
GROUP BY Sex, Pclass
ORDER BY survival_rate DESC;
  sql_db_query_checker (call_wrS0PfFluXyXSJTdY3D9d41B)
 Call ID: call_wr

In [16]:
agent_run("DB에서 컬럼은 어떤게 있어?", mode='stream')

================================ Human Message =================================

DB에서 컬럼은 어떤게 있어?
================================== Ai Message ==================================
Tool Calls:
  sql_db_list_tables (call_9X2USe7KQSqIQSVQg1eXnmXS)
 Call ID: call_9X2USe7KQSqIQSVQg1eXnmXS
  Args:
================================= Tool Message =================================
Name: sql_db_list_tables

passengers
================================== Ai Message ==================================
Tool Calls:
  sql_db_schema (call_liYHaYMsIhautQh9dxmq3KXL)
 Call ID: call_liYHaYMsIhautQh9dxmq3KXL
  Args:
    table_names: passengers
================================= Tool Message =================================
Name: sql_db_schema


CREATE TABLE passengers (
	"PassengerId" INTEGER, 
	"Survived" INTEGER, 
	"Pclass" INTEGER, 
	"Name" TEXT, 
	"Sex" TEXT, 
	"Age" REAL, 
	"SibSp" INTEGER, 
	"Parch" INTEGER, 
	"Ticket" TEXT, 
	"Fare" REAL, 
	"Cabin" TEXT, 
	"Embarked" TEXT
)

/*
3 rows from passengers t

In [17]:
agent_run("제일 마지막에 있는건 뭐야?", mode='stream')

================================ Human Message =================================

제일 마지막에 있는건 뭐야?
================================== Ai Message ==================================

제일 마지막 컬럼은 Embarked입니다. 

- Embarked: 승선 항구를 나타내는 텍스트 컬럼
  - 값 예: 'C' = Cherbourg, 'Q' = Queenstown, 'S' = Southampton
  - 간혹 NULL(결측치)인 경우도 있습니다.
  
샘플 데이터의 예시로는 S, C, S와 같은 값이 보였습니다. 필요하시면 각 값의 개수나 비율도 빠르게 확인해 드리겠습니다.


## 3) 테이블이 여러개인 DB 읽기

https://www.w3schools.com/sql/trysql.asp?filename=trysql_select_all

In [18]:
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import SQLDatabaseToolkit

db_path = '../data/w3schools'
with open(f'{db_path}/metadata.txt', 'r') as f:
    metadata = f.read()
    
db = SQLDatabase.from_uri(f"sqlite:///{db_path}/w3schools.db")
print(f"Dialect: {db.dialect}")
table = db.get_usable_table_names()[0]
print(f"Available tables: {table}")
query = f"SELECT * FROM {table} LIMIT 5;"
print(f'Sample output: {db.run(query)}')

toolkit = SQLDatabaseToolkit(db=db, llm=llm)

tools = toolkit.get_tools()

Dialect: sqlite
Available tables: Categories
Sample output: [(1, 'Beverages', 'Soft drinks, coffees, teas, beers, and ales'), (2, 'Condiments', 'Sweet and savory sauces, relishes, spreads, and seasonings'), (3, 'Confections', 'Desserts, candies, and sweet breads'), (4, 'Dairy Products', 'Cheeses'), (5, 'Grains/Cereals', 'Breads, crackers, pasta, and cereal')]


In [19]:
system_prompt = """
당신은 SQL 데이터베이스와 상호작용하도록 설계된 에이전트입니다.
입력된 질문을 바탕으로, 문법적으로 올바른 {dialect} 쿼리를 작성하고 실행한 뒤,
그 결과를 살펴보고 답변을 생성하세요.

사용하는 Database에 대한 정보는 다름과 같습니다.
## 메타데이터
# {metadata}

## 규칙:
1. 사용자가 특정한 예시 개수를 지정하지 않았다면, 항상 최대 {top_k}개의 결과만 반환하도록 쿼리에 제한을 두세요.
2. 결과는 데이터베이스에서 가장 흥미로운 예시가 위로 오도록 관련 있는 컬럼을 기준으로 정렬할 수 있습니다.
3. 절대로 모든 컬럼을 조회(SELECT *)하지 마세요. 질문과 관련된 컬럼만 선택해 쿼리를 작성해야 합니다.
4. 쿼리를 실행하기 전에 반드시 두 번 이상 검토하세요. 실행 중 오류가 발생하면, 쿼리를 수정하여 다시 시도해야 합니다.
5. INSERT, UPDATE, DELETE, DROP 등 데이터를 변경하는 DML 문장은 절대 실행하지 마세요.
6. 작업을 시작할 때는 반드시 데이터베이스 내의 테이블 목록을 먼저 확인하세요. 이 과정을 건너뛰지 마세요.
7. 그 후, 질문과 가장 관련이 있는 테이블의 스키마를 조회하여 구조를 이해한 뒤 쿼리를 작성하세요.
""".format(
    dialect=db.dialect,
    metadata=metadata,
    top_k=5,
)

from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    llm,
    tools,
    system_prompt=system_prompt,
    checkpointer=InMemorySaver(), 
)

In [20]:
def agent_run(question, mode = 'invoke'):
    cfg = {'configurable': {'thread_id': 'user-1'}}
    if mode == 'stream':
        for step in agent.stream(
            {"messages": [{"role": "user", "content": question}]},
            stream_mode="values",
            config=cfg,
        ):
            step["messages"][-1].pretty_print()
    elif mode == 'invoke':
        result = agent.invoke({"messages": [{"role": "user", "content": question}]}, config=cfg)
        return result.get("messages", [])[-1].content
    else:
        raise ValueError("Invalid mode")

In [21]:
agent_run("이 DB에는 어떤 테이블이 있어?", mode='stream')

================================ Human Message =================================

이 DB에는 어떤 테이블이 있어?
================================== Ai Message ==================================
Tool Calls:
  sql_db_list_tables (call_9duLKWbFvPyEKCJ5Xg0jM2L3)
 Call ID: call_9duLKWbFvPyEKCJ5Xg0jM2L3
  Args:
================================= Tool Message =================================
Name: sql_db_list_tables

Categories, Customers, Employees, OrderDetails, Orders, Products, Shippers, Suppliers
================================== Ai Message ==================================

다음과 같이 이 DB에 있는 테이블이 있습니다:

- Categories
- Customers
- Employees
- OrderDetails
- Orders
- Products
- Shippers
- Suppliers

각 테이블의 간략한 용도
- Categories: 제품의 분류 메타 정보
- Customers: 고객(회사)의 기본 정보
- Suppliers: 공급사 기본 정보
- Employees: 주문을 처리한 직원 정보
- Shippers: 배송사 정보
- Products: 판매 제품의 기본 정보
- Orders: 주문 헤더(고객이 생성한 주문)의 기본 속성
- OrderDetails: 주문에 포함된 개별 품목(라인 아이템) 정보

원하시면 특정 테이블의 스키마(컬럼 정의 및 데이터 타입, 관계)를 확인해 드리겠습니다. 예를 들어 "Categories

In [22]:
agent_run("각 테이블에 대해서 짧게 설명해줘", mode='stream')

================================ Human Message =================================

각 테이블에 대해서 짧게 설명해줘
================================== Ai Message ==================================

다음은 각 테이블에 대한 간단한 설명입니다.

- Categories: 제품의 분류 메타데이터를 저장. 주요 컬럼은 CategoryID, CategoryName, Description. Products와 1:N 관계.

- Customers: 고객(회사) 기본 정보 저장. 주요 컬럼은 CustomerID, CustomerName, ContactName, Address, City, PostalCode, Country. Orders와 1:N 관계.

- Suppliers: 공급사 기본 정보 저장. 주요 컬럼은 SupplierID, SupplierName, ContactName, Address, City, PostalCode, Country, Phone. Products와 1:N 관계.

- Employees: 주문을 처리한 직원 정보 저장. 주요 컬럼은 EmployeeID, LastName, FirstName, BirthDate, Photo, Notes. Orders와 1:N 관계.

- Shippers: 배송사 정보 저장. 주요 컬럼은 ShipperID, ShipperName, Phone. Orders와 1:N 관계.

- Products: 판매 제품의 기본 정보 저장. 주요 컬럼은 ProductID, ProductName, SupplierID, CategoryID, Unit, Price. Suppliers와 Categories에 FK; OrderDetails와 1:N 관계(주문 상세의 ProductID로 연결).

- Orders: 주문 헤더(고객이 생성한 주문)의 기본 속성 저장. 주요 컬럼은 OrderID, CustomerID, Emp

In [23]:
agent_run("ID가 2번 고객의 이름이 뭐야?", mode='stream')

================================ Human Message =================================

ID가 2번 고객의 이름이 뭐야?
================================== Ai Message ==================================
Tool Calls:
  sql_db_query_checker (call_3YdflqfDEdOxt05lX4cY5Dff)
 Call ID: call_3YdflqfDEdOxt05lX4cY5Dff
  Args:
    query: SELECT CustomerName FROM Customers WHERE CustomerID = 2 LIMIT 1;
================================= Tool Message =================================
Name: sql_db_query_checker

SELECT CustomerName FROM Customers WHERE CustomerID = 2 LIMIT 1;
================================== Ai Message ==================================
Tool Calls:
  sql_db_query (call_9xkKSWJNaAM0bmPIQL41uEGh)
 Call ID: call_9xkKSWJNaAM0bmPIQL41uEGh
  Args:
    query: SELECT CustomerName FROM Customers WHERE CustomerID = 2 LIMIT 1;
================================= Tool Message =================================
Name: sql_db_query

[('Ana Trujillo Emparedados y helados',)]
================================== Ai Message 

In [24]:
agent_run("그 사람이 구매한 물건 리스트를 보여줘", mode='stream')

================================ Human Message =================================

그 사람이 구매한 물건 리스트를 보여줘
================================== Ai Message ==================================
Tool Calls:
  sql_db_schema (call_1yO5yA3n50a6knIqInucr30d)
 Call ID: call_1yO5yA3n50a6knIqInucr30d
  Args:
    table_names: Customers, Orders, OrderDetails, Products
================================= Tool Message =================================
Name: sql_db_schema


CREATE TABLE "Customers" (
	"CustomerID" INTEGER, 
	"CustomerName" TEXT NOT NULL, 
	"ContactName" TEXT, 
	"Address" TEXT, 
	"City" TEXT, 
	"PostalCode" TEXT, 
	"Country" TEXT, 
	PRIMARY KEY ("CustomerID")
)

/*
3 rows from Customers table:
CustomerID	CustomerName	ContactName	Address	City	PostalCode	Country
1	Alfreds Futterkiste	Maria Anders	Obere Str. 57	Berlin	12209	Germany
2	Ana Trujillo Emparedados y helados	Ana Trujillo	Avda. de la Constitución 2222	México D.F.	05021	Mexico
3	Antonio Moreno Taquería	Antonio Moreno	Mataderos 2312	México

In [25]:
agent_run("가장 많은 수량이 팔린 물건은 뭐야?", mode='stream')

================================ Human Message =================================

가장 많은 수량이 팔린 물건은 뭐야?
================================== Ai Message ==================================
Tool Calls:
  sql_db_query_checker (call_DEJofcdF2wat4NYMi0axskiG)
 Call ID: call_DEJofcdF2wat4NYMi0axskiG
  Args:
    query: SELECT P.ProductName, SUM(OD.Quantity) AS TotalQuantity
FROM OrderDetails OD
JOIN Products P ON OD.ProductID = P.ProductID
GROUP BY P.ProductName
ORDER BY TotalQuantity DESC
LIMIT 1;
================================= Tool Message =================================
Name: sql_db_query_checker

SELECT P.ProductName, SUM(OD.Quantity) AS TotalQuantity
FROM OrderDetails OD
JOIN Products P ON OD.ProductID = P.ProductID
GROUP BY P.ProductName
ORDER BY TotalQuantity DESC
LIMIT 1;
================================== Ai Message ==================================
Tool Calls:
  sql_db_query (call_sSz8rkWvFCQKtHrZnFs3oWeq)
 Call ID: call_sSz8rkWvFCQKtHrZnFs3oWeq
  Args:
    query: SELECT P.Product